# MimicryDB-Auto: Statistical Analysis Pipeline

**Paper:** MimicryDB-Auto: A Curated Multi-Pathogen Dataset of Structurally Validated Pathogen-Host Peptide Pairs  

---

## Notebook Structure
1. Setup & Data Loading
2. Data Preparation
3. Core Statistical Analyses (Section 3.2)
4. Original RF Classifier — Y vs N at 2.0 Å (Section 3.3)
5. Strong vs Weak Mimic RF — within Y-class at 1.0 Å (Section 3.3 / 4.3)
6. Threshold Sensitivity Analysis (Section 3.4)
7. Figures

---
## SECTION 1: Setup & Data Loading

In [5]:
# ─────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, mannwhitneyu, norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
                             accuracy_score, classification_report,
                             confusion_matrix, roc_curve)
from matplotlib.patches import Patch

print('All libraries loaded successfully')

All libraries loaded successfully


In [6]:
# ── Load dataset ───────────────────────────────────────────────────
df = pd.read_excel('/content/ML targets (10).xlsx', sheet_name='Sheet1')
print('Shape:', df.shape)
print('\nLabel counts:')
print(df['RMSD_Mimic_Target (Y)'].value_counts())
print('\nColumns:', df.columns.tolist())

Shape: (399, 25)

Label counts:
RMSD_Mimic_Target (Y)
Y    262
N    137
Name: count, dtype: int64

Columns: ['Organism', 'Protein', 'Position', 'HLA Haplotype', 'Pathogen Peptide', 'pathogen length', '%Rank_EL(X)', 'Aff(nM)(X)', 'Immunogenicity', 'Type of MHC', 'Human_match', 'BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)', 'Identical aa', 'Positions', 'Human Peptide', 'Human length', 'Alignment Length (Structure)', 'Structural RMSD', 'TM-align score (Human chain 2)', 'Structural alignment coverage %', 'RMSD_Mimic_Target (Y)', 'BLOSUM80_per_residue ', 'Alignment_coverage_sequence']


---
## SECTION 2: Data Preparation

This section comes before anything else. It converts types, computes derived features, and defines all class subgroups used throughout.

In [7]:
# ── Type conversion + derived features ─────────────────────────────
# Converting all numeric columns from object type
for col in ['BLOSUM80 score', 'Identity percentage', 'Alignment length (Sequence)',
            'Identical aa', 'pathogen length', 'Human length', '%Rank_EL(X)',
            'Aff(nM)(X)', 'Structural RMSD', 'TM-align score (Human chain 2)',
            'Structural alignment coverage %', 'Alignment Length (Structure)']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Derived feature 1: BLOSUM80 normalised by alignment length
df['BLOSUM80_per_residue'] = df['BLOSUM80 score'] / df['Alignment length (Sequence)']

# Derived feature 2: what fraction of pathogen peptide was covered by BLAST
df['Alignment_coverage_sequence'] = (
    df['Alignment length (Sequence)'] / df['pathogen length'] * 100
).clip(upper=100)

# Fill NaN with 0 — these are genuine no-BLAST-hit entries, not missing data
df['BLOSUM80_per_residue'] = df['BLOSUM80_per_residue'].fillna(0)
df['Alignment_coverage_sequence'] = df['Alignment_coverage_sequence'].fillna(0)

print('Conversion complete. Shape:', df.shape)
print('Missing values remaining:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Conversion complete. Shape: (399, 26)
Missing values remaining:
Series([], dtype: int64)


In [8]:
# ── Defining the class subgroups ─────────────────────────────────────
# 2.0 Å binary classification
Y_group = df[df['RMSD_Mimic_Target (Y)'] == 'Y']   # n=262, RMSD < 2.0 Å
N_group = df[df['RMSD_Mimic_Target (Y)'] == 'N']   # n=137, RMSD >= 2.0 Å

# At 1.0 Å
strong_mimics = df[df['Structural RMSD'] < 1.0]
weak_mimics   = df[(df['Structural RMSD'] >= 1.0) & (df['Structural RMSD'] < 2.0)]
non_mimics    = df[df['Structural RMSD'] >= 2.0]

# Consistent colour scheme used throughout all figures
strong_color = '#1f78b4'   # deep blue
weak_color   = '#e6ab02'   # golden
non_color    = '#d95f02'   # orange-red

print(f'Y_group (RMSD < 2.0 Å):  n={len(Y_group)}')
print(f'N_group (RMSD >= 2.0 Å): n={len(N_group)}')
print(f'Strong mimics (< 1.0 Å): n={len(strong_mimics)}')
print(f'Weak mimics (1.0-2.0 Å): n={len(weak_mimics)}')
print(f'Non-mimics  (>= 2.0 Å):  n={len(non_mimics)}')

Y_group (RMSD < 2.0 Å):  n=262
N_group (RMSD >= 2.0 Å): n=137
Strong mimics (< 1.0 Å): n=159
Weak mimics (1.0-2.0 Å): n=103
Non-mimics  (>= 2.0 Å):  n=137
